In [1]:
# ============================================================
# Minimal setup for Step 2
# Upload config.py and Step 1 zip separately if needed.
# ============================================================

import os
import sys
import shutil
import zipfile
import importlib
from pathlib import Path
from google.colab import files

# 1. Create project structure
PILOT_ROOT = "/content/pilot_3"
SCRIPTS_DIR = os.path.join(PILOT_ROOT, "scripts")
DATA_DIR = os.path.join(PILOT_ROOT, "data")

os.makedirs(PILOT_ROOT, exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Make pilot_3 importable as a package.
with open(os.path.join(PILOT_ROOT, "__init__.py"), "w", encoding="utf-8") as f:
    f.write("")

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# 2. Upload config.py
print("Upload config.py")
uploaded_config = files.upload()

config_upload_name = None
for name in uploaded_config.keys():
    if name.startswith("config") and name.endswith(".py"):
        config_upload_name = name
        break

if config_upload_name is None:
    raise FileNotFoundError(
        f"No config*.py file was uploaded. Uploaded files: {list(uploaded_config.keys())}"
    )

shutil.copy(
    os.path.join("/content", config_upload_name),
    os.path.join(PILOT_ROOT, "config.py"),
)

# 3. Load config after upload
import pilot_3.config as cfg
importlib.reload(cfg)

os.makedirs(cfg.STEP1_DIR, exist_ok=True)

if os.path.exists(cfg.STEP2_DIR):
    shutil.rmtree(cfg.STEP2_DIR)
os.makedirs(cfg.STEP2_DIR, exist_ok=True)

print("Copied config.py to:", os.path.join(PILOT_ROOT, "config.py"))
print("STEP1_DIR:", cfg.STEP1_DIR)
print("STEP2_DIR:", cfg.STEP2_DIR)
print("SCENES:", cfg.SCENES)

# 4. Upload Step 1 zip
print("Upload Step 1 zip, for example pilot3_step1_chunk_ground_truth.zip")
uploaded_zip = files.upload()

zip_candidates = [
    name for name in uploaded_zip.keys()
    if name.endswith(".zip") and "step1" in name.lower()
]

if not zip_candidates:
    raise FileNotFoundError(
        "Please upload a Step 1 zip file whose name contains 'step1'."
    )

zip_path = os.path.join("/content", zip_candidates[0])

# 5. Unzip Step 1 output
tmp_extract_dir = "/content/_step1_extract_tmp"

if os.path.exists(tmp_extract_dir):
    shutil.rmtree(tmp_extract_dir)

os.makedirs(tmp_extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(tmp_extract_dir)

# 6. Copy FloorPlan json files into cfg.STEP1_DIR
json_files = list(Path(tmp_extract_dir).rglob("*.json"))

scene_json_files = [
    jf for jf in json_files
    if jf.name.startswith("FloorPlan") and jf.name.endswith(".json")
]

if not scene_json_files:
    raise FileNotFoundError("No FloorPlan*.json files found in the Step 1 zip.")

for jf in scene_json_files:
    shutil.copy(str(jf), os.path.join(cfg.STEP1_DIR, jf.name))

print("\nSetup complete.")
print("config.py -> /content/pilot_3/config.py")
print("Step 1 json files ->", cfg.STEP1_DIR)

for name in sorted(os.listdir(cfg.STEP1_DIR)):
    print("-", name)

Upload config.py


Saving config.py to config.py
Copied config.py to: /content/pilot_3/config.py
STEP1_DIR: /content/pilot_3/data/step1_chunk_ground_truth
STEP2_DIR: /content/pilot_3/data/step2_chunk_relations
SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6', 'FloorPlan7']
Upload Step 1 zip, for example pilot3_step1_chunk_ground_truth.zip


Saving pilot3_step1_chunk_ground_truth.zip to pilot3_step1_chunk_ground_truth.zip

Setup complete.
config.py -> /content/pilot_3/config.py
Step 1 json files -> /content/pilot_3/data/step1_chunk_ground_truth
- FloorPlan1.json
- FloorPlan2.json
- FloorPlan3.json
- FloorPlan4.json
- FloorPlan5.json
- FloorPlan6.json


In [2]:
import json
import math
import os
import random
import sys
import importlib
from collections import Counter
from typing import Any, Dict, List, Tuple, Optional, Set

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

import pilot_3.config as cfg
importlib.reload(cfg)

from pilot_3.config import (
    STEP1_DIR,
    STEP2_DIR,
    SCENES,
    RANDOM_SEED,

    STRUCTURAL_TYPES,
    SURFACE_TYPES,
    CONTAINER_LIKE_TYPES,

    INVERSE_RELATION_MAP,
    INVERSE_RELATION_GROUPS,

    LEFT_MARGIN,
    ABOVE_Y_MARGIN,
    NEAR_THRESHOLD,

    LEFT_REQUIRE_Z_OVERLAP_OR_NEAR,
    LEFT_MAX_Z_CENTER_DISTANCE,

    MAX_NEAR_RELATIONS_PER_CLUSTER,
    MAX_TEXT_CANDIDATE_RELATIONS_PER_CLUSTER,

    BUILD_GEOMETRY_PAIR_RECORDS,
    MAX_GEOMETRY_PAIRS_PER_CLUSTER,
)

random.seed(RANDOM_SEED)

os.makedirs(STEP2_DIR, exist_ok=True)

print("STEP1_DIR:", STEP1_DIR)
print("STEP2_DIR:", STEP2_DIR)
print("SCENES:", SCENES)

print("\nStep 2 config:")
print("SURFACE_TYPES:", SURFACE_TYPES)
print("CONTAINER_LIKE_TYPES:", CONTAINER_LIKE_TYPES)
print("STRUCTURAL_TYPES:", STRUCTURAL_TYPES)
print("INVERSE_RELATION_MAP:", INVERSE_RELATION_MAP)
print("LEFT_MARGIN:", LEFT_MARGIN)
print("ABOVE_Y_MARGIN:", ABOVE_Y_MARGIN)
print("NEAR_THRESHOLD:", NEAR_THRESHOLD)
print("LEFT_REQUIRE_Z_OVERLAP_OR_NEAR:", LEFT_REQUIRE_Z_OVERLAP_OR_NEAR)
print("LEFT_MAX_Z_CENTER_DISTANCE:", LEFT_MAX_Z_CENTER_DISTANCE)
print("MAX_NEAR_RELATIONS_PER_CLUSTER:", MAX_NEAR_RELATIONS_PER_CLUSTER)
print("MAX_TEXT_CANDIDATE_RELATIONS_PER_CLUSTER:", MAX_TEXT_CANDIDATE_RELATIONS_PER_CLUSTER)
print("BUILD_GEOMETRY_PAIR_RECORDS:", BUILD_GEOMETRY_PAIR_RECORDS)
print("MAX_GEOMETRY_PAIRS_PER_CLUSTER:", MAX_GEOMETRY_PAIRS_PER_CLUSTER)

print("\nStep 1 files:")
for name in sorted(os.listdir(STEP1_DIR)):
    print("-", name)

STEP1_DIR: /content/pilot_3/data/step1_chunk_ground_truth
STEP2_DIR: /content/pilot_3/data/step2_chunk_relations
SCENES: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6', 'FloorPlan7']

Step 2 config:
SURFACE_TYPES: {'DiningTable', 'Desk', 'SideTable', 'StoveBurner', 'Sink', 'TableTop', 'CounterTop', 'Shelf'}
CONTAINER_LIKE_TYPES: {'Plate', 'Microwave', 'Pan', 'Pot', 'Mug', 'Safe', 'Drawer', 'Cup', 'Bowl', 'Cabinet', 'Fridge'}
STRUCTURAL_TYPES: {'Microwave', 'Drawer', 'Sink', 'CoffeeMachine', 'TableTop', 'Cabinet', 'TVStand', 'Shelf', 'DiningTable', 'Toaster', 'Safe', 'Desk', 'SideTable', 'StoveBurner', 'CounterTop', 'Fridge'}
INVERSE_RELATION_MAP: {'in': 'contains', 'contains': 'in', 'on': 'supports', 'supports': 'on', 'left_of': 'right_of', 'right_of': 'left_of', 'above': 'below', 'below': 'above', 'near': 'near', 'none': 'none'}
LEFT_MARGIN: 0.05
ABOVE_Y_MARGIN: 0.03
NEAR_THRESHOLD: 0.5
LEFT_REQUIRE_Z_OVERLAP_OR_NEAR: True
LEFT_MAX_Z_CENTER_DISTANC

In [3]:
def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path: str, data: Dict[str, Any]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def build_object_index(objects: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    return {
        obj["objectId"]: obj
        for obj in objects
        if obj.get("objectId") is not None
    }


def is_structural_object(obj: Dict[str, Any]) -> bool:
    return obj.get("objectType") in STRUCTURAL_TYPES


def is_same_object_type(obj_a: Dict[str, Any], obj_b: Dict[str, Any]) -> bool:
    return obj_a.get("objectType") == obj_b.get("objectType")


def get_position(obj: Dict[str, Any]) -> Dict[str, float]:
    pos = obj.get("position", {}) or {}
    return {
        "x": float(pos.get("x", 0.0)),
        "y": float(pos.get("y", 0.0)),
        "z": float(pos.get("z", 0.0)),
    }


def get_extent(obj: Dict[str, Any]) -> Dict[str, float]:
    """
    Compute object extent from bbox.center and bbox.size.
    Fall back to position with zero size if bbox is missing.
    """
    bbox = obj.get("bbox", {}) or {}
    center = bbox.get("center", {}) or obj.get("position", {}) or {}
    size = bbox.get("size", {}) or {}

    cx = float(center.get("x", 0.0))
    cy = float(center.get("y", 0.0))
    cz = float(center.get("z", 0.0))

    sx = float(size.get("x", 0.0))
    sy = float(size.get("y", 0.0))
    sz = float(size.get("z", 0.0))

    return {
        "x_min": cx - sx / 2.0,
        "x_max": cx + sx / 2.0,
        "y_min": cy - sy / 2.0,
        "y_max": cy + sy / 2.0,
        "z_min": cz - sz / 2.0,
        "z_max": cz + sz / 2.0,
    }


def overlap_1d(a_min: float, a_max: float, b_min: float, b_max: float) -> float:
    return min(a_max, b_max) - max(a_min, b_min)


def has_overlap_1d(
    a_min: float,
    a_max: float,
    b_min: float,
    b_max: float,
    margin: float = 0.0,
) -> bool:
    return overlap_1d(a_min, a_max, b_min, b_max) > margin


def horizontal_distance(obj_a: Dict[str, Any], obj_b: Dict[str, Any]) -> float:
    pa = get_position(obj_a)
    pb = get_position(obj_b)

    dx = pa["x"] - pb["x"]
    dz = pa["z"] - pb["z"]

    return math.sqrt(dx * dx + dz * dz)


print("IO and geometry helpers loaded.")

IO and geometry helpers loaded.


In [4]:
def center_delta(subject_obj: Dict[str, Any], object_obj: Dict[str, Any]) -> Dict[str, float]:
    """
    Continuous relative geometry label for Experiment B.

    Direction:
        delta = subject - object
    """
    ps = get_position(subject_obj)
    po = get_position(object_obj)

    dx = ps["x"] - po["x"]
    dy = ps["y"] - po["y"]
    dz = ps["z"] - po["z"]

    return {
        "dx": dx,
        "dy": dy,
        "dz": dz,
        "horizontal_distance": math.sqrt(dx * dx + dz * dz),
        "euclidean_distance": math.sqrt(dx * dx + dy * dy + dz * dz),
    }


def bbox_geometry(subject_obj: Dict[str, Any], object_obj: Dict[str, Any]) -> Dict[str, Any]:
    es = get_extent(subject_obj)
    eo = get_extent(object_obj)

    x_overlap = overlap_1d(es["x_min"], es["x_max"], eo["x_min"], eo["x_max"])
    y_overlap = overlap_1d(es["y_min"], es["y_max"], eo["y_min"], eo["y_max"])
    z_overlap = overlap_1d(es["z_min"], es["z_max"], eo["z_min"], eo["z_max"])

    return {
        "subject_bottom_minus_object_top": es["y_min"] - eo["y_max"],
        "subject_top_minus_object_bottom": es["y_max"] - eo["y_min"],
        "x_overlap": x_overlap,
        "y_overlap": y_overlap,
        "z_overlap": z_overlap,
        "has_x_overlap": x_overlap > 0,
        "has_y_overlap": y_overlap > 0,
        "has_z_overlap": z_overlap > 0,
    }


def build_geometry_label(
    subject_obj: Dict[str, Any],
    object_obj: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "center_delta": center_delta(subject_obj, object_obj),
        "bbox_geometry": bbox_geometry(subject_obj, object_obj),
    }


print("Experiment B geometry label helpers loaded.")

Experiment B geometry label helpers loaded.


In [5]:
def infer_topological_relation(child_obj: Dict[str, Any], parent_obj: Dict[str, Any]) -> str:
    """
    Infer whether child-parent relation should be verbalized as on or in.
    """
    parent_type = parent_obj.get("objectType")

    if parent_type in SURFACE_TYPES:
        return "on"

    return "in"


def is_left_of(
    obj_a: Dict[str, Any],
    obj_b: Dict[str, Any],
    margin: float = LEFT_MARGIN,
) -> bool:
    ea = get_extent(obj_a)
    eb = get_extent(obj_b)

    x_separated = ea["x_max"] < eb["x_min"] - margin

    if not x_separated:
        return False

    if not LEFT_REQUIRE_Z_OVERLAP_OR_NEAR:
        return True

    z_overlap = has_overlap_1d(ea["z_min"], ea["z_max"], eb["z_min"], eb["z_max"])
    za = (ea["z_min"] + ea["z_max"]) / 2.0
    zb = (eb["z_min"] + eb["z_max"]) / 2.0
    z_center_close = abs(za - zb) <= LEFT_MAX_Z_CENTER_DISTANCE

    return z_overlap or z_center_close


def is_above(
    obj_a: Dict[str, Any],
    obj_b: Dict[str, Any],
    y_margin: float = ABOVE_Y_MARGIN,
) -> bool:
    ea = get_extent(obj_a)
    eb = get_extent(obj_b)

    x_overlap = has_overlap_1d(ea["x_min"], ea["x_max"], eb["x_min"], eb["x_max"])
    z_overlap = has_overlap_1d(ea["z_min"], ea["z_max"], eb["z_min"], eb["z_max"])
    y_clear = ea["y_min"] > eb["y_max"] + y_margin

    return x_overlap and z_overlap and y_clear


def is_near(
    obj_a: Dict[str, Any],
    obj_b: Dict[str, Any],
    threshold: float = NEAR_THRESHOLD,
) -> bool:
    return horizontal_distance(obj_a, obj_b) <= threshold


print("Relation inference rules loaded.")

Relation inference rules loaded.


In [6]:
def should_keep_left_of(obj_a: Dict[str, Any], obj_b: Dict[str, Any]) -> bool:
    """
    Avoid structural-structural left/right relations.
    """
    if is_structural_object(obj_a) and is_structural_object(obj_b):
        return False
    return True


def should_keep_above(obj_a: Dict[str, Any], obj_b: Dict[str, Any]) -> bool:
    """
    Avoid structural-structural vertical relations.
    """
    if is_structural_object(obj_a) and is_structural_object(obj_b):
        return False

    if is_same_object_type(obj_a, obj_b) and is_structural_object(obj_a):
        return False

    return True


def should_keep_near_as_subject(obj_a: Dict[str, Any], obj_b: Dict[str, Any]) -> bool:
    """
    Keep natural near descriptions:
    - small -> structural
    - small -> small

    Avoid:
    - structural -> structural
    - structural -> small
    """
    a_structural = is_structural_object(obj_a)
    b_structural = is_structural_object(obj_b)

    if a_structural and b_structural:
        return False

    if a_structural and not b_structural:
        return False

    return True


def should_mark_verbalizable(rel: Dict[str, Any]) -> bool:
    relation = rel["relation"]
    subj_type = rel["subject_type"]
    obj_type = rel["object_type"]

    subj_structural = subj_type in STRUCTURAL_TYPES
    obj_structural = obj_type in STRUCTURAL_TYPES

    if rel["relation_family"] == "topological":
        return True

    if relation in {"left_of", "right_of", "above", "below"}:
        if subj_structural and obj_structural:
            return False

    if relation == "near":
        if subj_structural and obj_structural:
            return False

    return True


print("Relation filtering rules loaded.")

Relation filtering rules loaded.


In [7]:
def make_relation_record(
    scene: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
    subject_obj: Dict[str, Any],
    relation: str,
    object_obj: Dict[str, Any],
    family: str,
    evidence: Optional[Dict[str, Any]] = None,
    is_inverse_relation: bool = False,
    derived_from_relation: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Relation record for Experiment A.
    Also includes continuous geometry labels for Experiment B.
    """
    record = {
        "scene": scene,

        "chunk_id": chunk["chunk_id"],
        "cluster_id": cluster["cluster_id"],
        "source_chunk_id": cluster.get("source_chunk_id", chunk["chunk_id"]),
        "grid_i": chunk.get("grid_i"),
        "grid_j": chunk.get("grid_j"),
        "cluster_anchor_object_ids": cluster.get("anchor_object_ids", []),
        "cluster_num_objects": cluster.get("num_objects"),

        "subject_id": subject_obj["objectId"],
        "subject_type": subject_obj["objectType"],

        "relation": relation,

        "object_id": object_obj["objectId"],
        "object_type": object_obj["objectType"],

        "relation_family": family,
        "candidate_for_text": True,
        "verbalizable": True,

        "is_inverse_relation": is_inverse_relation,
        "derived_from_relation": derived_from_relation,

        "evidence": evidence or {},

        # For Experiment B
        "geometry_label": build_geometry_label(subject_obj, object_obj),
    }

    record["verbalizable"] = should_mark_verbalizable(record)
    record["candidate_for_text"] = record["verbalizable"]

    return record


def relation_priority(rel: Dict[str, Any]) -> int:
    """
    Higher = more useful / stronger relation.
    """
    r = rel["relation"]

    if rel["relation_family"] == "topological":
        return 100

    if r in {"above", "below"}:
        return 80

    if r in {"left_of", "right_of"}:
        return 70

    if r == "near":
        return 40

    return 10


print("Relation record construction loaded.")

Relation record construction loaded.


In [8]:
def build_topological_relations(
    scene: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
    objects: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Recover in/on from parentReceptacles within the current cluster.
    """
    obj_index = build_object_index(objects)
    relations = []

    for obj in objects:
        for parent_id in obj.get("parentReceptacles", []) or []:
            if parent_id not in obj_index:
                continue

            parent_obj = obj_index[parent_id]
            relation = infer_topological_relation(obj, parent_obj)

            rel = make_relation_record(
                scene=scene,
                chunk=chunk,
                cluster=cluster,
                subject_obj=obj,
                relation=relation,
                object_obj=parent_obj,
                family="topological",
                evidence={
                    "rule": "parentReceptacles",
                    "parent_id": parent_id,
                    "parent_type": parent_obj.get("objectType"),
                },
            )
            relations.append(rel)

    return relations


print("Topological relation builder loaded.")

Topological relation builder loaded.


In [9]:
def build_geometric_relations(
    scene: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
    objects: List[Dict[str, Any]],
    topological_relations: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """
    Build left_of / above / near relations inside one cluster.
    """
    relations = []

    topo_unordered_pairs = set()
    for rel in topological_relations:
        pair = tuple(sorted([rel["subject_id"], rel["object_id"]]))
        topo_unordered_pairs.add(pair)

    strong_relations = []
    near_candidates = []

    n = len(objects)

    for i in range(n):
        for j in range(i + 1, n):
            obj_a = objects[i]
            obj_b = objects[j]

            a_id = obj_a["objectId"]
            b_id = obj_b["objectId"]
            unordered_pair = tuple(sorted([a_id, b_id]))

            if unordered_pair in topo_unordered_pairs:
                continue

            chosen_relation = None

            # Priority 1: above
            if is_above(obj_a, obj_b) and should_keep_above(obj_a, obj_b):
                ea = get_extent(obj_a)
                eb = get_extent(obj_b)
                chosen_relation = make_relation_record(
                    scene=scene,
                    chunk=chunk,
                    cluster=cluster,
                    subject_obj=obj_a,
                    relation="above",
                    object_obj=obj_b,
                    family="geometric",
                    evidence={
                        "rule": "vertical_clearance_with_xz_overlap",
                        "y_clearance": ea["y_min"] - eb["y_max"],
                        "x_overlap": overlap_1d(ea["x_min"], ea["x_max"], eb["x_min"], eb["x_max"]),
                        "z_overlap": overlap_1d(ea["z_min"], ea["z_max"], eb["z_min"], eb["z_max"]),
                        "y_margin": ABOVE_Y_MARGIN,
                    },
                )

            elif is_above(obj_b, obj_a) and should_keep_above(obj_b, obj_a):
                eb = get_extent(obj_b)
                ea = get_extent(obj_a)
                chosen_relation = make_relation_record(
                    scene=scene,
                    chunk=chunk,
                    cluster=cluster,
                    subject_obj=obj_b,
                    relation="above",
                    object_obj=obj_a,
                    family="geometric",
                    evidence={
                        "rule": "vertical_clearance_with_xz_overlap",
                        "y_clearance": eb["y_min"] - ea["y_max"],
                        "x_overlap": overlap_1d(eb["x_min"], eb["x_max"], ea["x_min"], ea["x_max"]),
                        "z_overlap": overlap_1d(eb["z_min"], eb["z_max"], ea["z_min"], ea["z_max"]),
                        "y_margin": ABOVE_Y_MARGIN,
                    },
                )

            # Priority 2: left_of
            if chosen_relation is None:
                if is_left_of(obj_a, obj_b) and should_keep_left_of(obj_a, obj_b):
                    ea = get_extent(obj_a)
                    eb = get_extent(obj_b)
                    chosen_relation = make_relation_record(
                        scene=scene,
                        chunk=chunk,
                        cluster=cluster,
                        subject_obj=obj_a,
                        relation="left_of",
                        object_obj=obj_b,
                        family="geometric",
                        evidence={
                            "rule": "x_separation",
                            "x_gap": eb["x_min"] - ea["x_max"],
                            "margin": LEFT_MARGIN,
                            "z_guard": LEFT_REQUIRE_Z_OVERLAP_OR_NEAR,
                        },
                    )

                elif is_left_of(obj_b, obj_a) and should_keep_left_of(obj_b, obj_a):
                    eb = get_extent(obj_b)
                    ea = get_extent(obj_a)
                    chosen_relation = make_relation_record(
                        scene=scene,
                        chunk=chunk,
                        cluster=cluster,
                        subject_obj=obj_b,
                        relation="left_of",
                        object_obj=obj_a,
                        family="geometric",
                        evidence={
                            "rule": "x_separation",
                            "x_gap": ea["x_min"] - eb["x_max"],
                            "margin": LEFT_MARGIN,
                            "z_guard": LEFT_REQUIRE_Z_OVERLAP_OR_NEAR,
                        },
                    )

            if chosen_relation is not None:
                strong_relations.append(chosen_relation)
                continue

            # Priority 3: near
            if is_near(obj_a, obj_b):
                dist = horizontal_distance(obj_a, obj_b)

                if should_keep_near_as_subject(obj_a, obj_b):
                    rel = make_relation_record(
                        scene=scene,
                        chunk=chunk,
                        cluster=cluster,
                        subject_obj=obj_a,
                        relation="near",
                        object_obj=obj_b,
                        family="geometric",
                        evidence={
                            "rule": "horizontal_distance",
                            "horizontal_distance": dist,
                            "threshold": NEAR_THRESHOLD,
                        },
                    )
                    near_candidates.append(rel)

                elif should_keep_near_as_subject(obj_b, obj_a):
                    rel = make_relation_record(
                        scene=scene,
                        chunk=chunk,
                        cluster=cluster,
                        subject_obj=obj_b,
                        relation="near",
                        object_obj=obj_a,
                        family="geometric",
                        evidence={
                            "rule": "horizontal_distance",
                            "horizontal_distance": dist,
                            "threshold": NEAR_THRESHOLD,
                        },
                    )
                    near_candidates.append(rel)

    near_candidates.sort(
        key=lambda r: r.get("evidence", {}).get("horizontal_distance", 999.0)
    )

    relations.extend(strong_relations)
    relations.extend(near_candidates[:MAX_NEAR_RELATIONS_PER_CLUSTER])

    return relations


print("Geometric relation builder loaded.")

Geometric relation builder loaded.


In [10]:
def deduplicate_relations(relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for rel in relations:
        key = (
            rel["scene"],
            rel["chunk_id"],
            rel["cluster_id"],
            rel["subject_id"],
            rel["relation"],
            rel["object_id"],
        )

        if key in seen:
            continue

        seen.add(key)
        deduped.append(rel)

    return deduped


def expand_inverse_relations(relations: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    expanded = list(relations)

    for rel in relations:
        forward_relation = rel["relation"]

        if forward_relation not in INVERSE_RELATION_MAP:
            continue

        inverse_relation = INVERSE_RELATION_MAP[forward_relation]

        inverse_record = dict(rel)

        inverse_record["subject_id"] = rel["object_id"]
        inverse_record["subject_type"] = rel["object_type"]
        inverse_record["relation"] = inverse_relation
        inverse_record["object_id"] = rel["subject_id"]
        inverse_record["object_type"] = rel["subject_type"]

        inverse_record["is_inverse_relation"] = True
        inverse_record["derived_from_relation"] = forward_relation

        inverse_record["evidence"] = {
            "source": "inverse_relation",
            "forward_relation": forward_relation,
            "forward_evidence": rel.get("evidence", {}),
        }

        # Invert center_delta numerically from the forward geometry label.
        forward_geom = rel.get("geometry_label", {})
        forward_delta = forward_geom.get("center_delta", {})

        inv_delta = {
            "dx": -float(forward_delta.get("dx", 0.0)),
            "dy": -float(forward_delta.get("dy", 0.0)),
            "dz": -float(forward_delta.get("dz", 0.0)),
            "horizontal_distance": float(forward_delta.get("horizontal_distance", 0.0)),
            "euclidean_distance": float(forward_delta.get("euclidean_distance", 0.0)),
        }

        inverse_record["geometry_label"] = {
            "center_delta": inv_delta,
            "bbox_geometry": forward_geom.get("bbox_geometry", {}),
            "note": "bbox_geometry copied from forward relation; center_delta inverted.",
        }

        inverse_record["verbalizable"] = should_mark_verbalizable(inverse_record)
        inverse_record["candidate_for_text"] = inverse_record["verbalizable"]

        expanded.append(inverse_record)

    return expanded


def find_inverse_group(relation: str) -> Optional[Tuple[str, ...]]:
    for group in INVERSE_RELATION_GROUPS:
        if relation in group:
            return tuple(sorted(group))
    return None


def compress_inverse_relations_randomly(
    relations: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Randomly keep one expression from inverse relation pairs.
    """
    grouped = {}
    result = []

    for rel in relations:
        relation = rel["relation"]
        inverse_group = find_inverse_group(relation)

        if inverse_group is None:
            result.append(rel)
            continue

        obj_pair = tuple(sorted([rel["subject_id"], rel["object_id"]]))
        group_key = (
            rel["scene"],
            rel["chunk_id"],
            rel["cluster_id"],
            inverse_group,
            obj_pair,
        )

        grouped.setdefault(group_key, []).append(rel)

    for group in grouped.values():
        group_sorted = sorted(
            group,
            key=lambda r: (
                -relation_priority(r),
                r["subject_id"],
                r["relation"],
                r["object_id"],
            )
        )
        result.append(random.choice(group_sorted))

    return result


def global_deduplicate_relations(
    relations: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Deduplicate across overlapping clusters.
    """
    grouped = {}

    for rel in relations:
        key = (
            rel["scene"],
            rel["subject_id"],
            rel["relation"],
            rel["object_id"],
        )
        grouped.setdefault(key, []).append(rel)

    selected = []

    for group in grouped.values():
        group_sorted = sorted(
            group,
            key=lambda r: (
                -relation_priority(r),
                int(r.get("cluster_num_objects") or 999),
                r["chunk_id"],
                r["cluster_id"],
                r["subject_id"],
                r["object_id"],
            )
        )
        selected.append(group_sorted[0])

    selected.sort(
        key=lambda r: (
            r["scene"],
            r["chunk_id"],
            r["cluster_id"],
            r["subject_type"],
            r["relation"],
            r["object_type"],
            r["subject_id"],
            r["object_id"],
        )
    )

    return selected


print("Dedup and inverse relation helpers loaded.")

Dedup and inverse relation helpers loaded.


In [11]:
def should_keep_geometry_pair(obj_a: Dict[str, Any], obj_b: Dict[str, Any]) -> bool:
    """
    Keep candidate pairs for continuous geometry regression.
    Avoid structural-structural pairs.
    """
    if is_structural_object(obj_a) and is_structural_object(obj_b):
        return False

    if obj_a["objectId"] == obj_b["objectId"]:
        return False

    return True


def make_geometry_pair_record(
    scene: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
    subject_obj: Dict[str, Any],
    object_obj: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "scene": scene,

        "chunk_id": chunk["chunk_id"],
        "cluster_id": cluster["cluster_id"],
        "source_chunk_id": cluster.get("source_chunk_id", chunk["chunk_id"]),
        "grid_i": chunk.get("grid_i"),
        "grid_j": chunk.get("grid_j"),
        "cluster_anchor_object_ids": cluster.get("anchor_object_ids", []),
        "cluster_num_objects": cluster.get("num_objects"),

        "subject_id": subject_obj["objectId"],
        "subject_type": subject_obj["objectType"],

        "object_id": object_obj["objectId"],
        "object_type": object_obj["objectType"],

        "pair_family": "geometry_pair",
        "candidate_for_geometry_regression": True,

        "geometry_label": build_geometry_label(subject_obj, object_obj),
    }


def build_geometry_pair_records(
    scene: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
    objects: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    if not BUILD_GEOMETRY_PAIR_RECORDS:
        return []

    pairs = []

    n = len(objects)

    for i in range(n):
        for j in range(n):
            if i == j:
                continue

            obj_a = objects[i]
            obj_b = objects[j]

            if not should_keep_geometry_pair(obj_a, obj_b):
                continue

            pair = make_geometry_pair_record(
                scene=scene,
                chunk=chunk,
                cluster=cluster,
                subject_obj=obj_a,
                object_obj=obj_b,
            )
            pairs.append(pair)

    pairs.sort(
        key=lambda p: p["geometry_label"]["center_delta"]["horizontal_distance"]
    )

    return pairs[:MAX_GEOMETRY_PAIRS_PER_CLUSTER]


def global_deduplicate_geometry_pairs(
    pairs: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    grouped = {}

    for pair in pairs:
        key = (
            pair["scene"],
            pair["subject_id"],
            pair["object_id"],
        )
        grouped.setdefault(key, []).append(pair)

    selected = []

    for group in grouped.values():
        group_sorted = sorted(
            group,
            key=lambda p: (
                float(p["geometry_label"]["center_delta"]["horizontal_distance"]),
                int(p.get("cluster_num_objects") or 999),
                p["chunk_id"],
                p["cluster_id"],
            )
        )
        selected.append(group_sorted[0])

    selected.sort(
        key=lambda p: (
            p["scene"],
            p["chunk_id"],
            p["cluster_id"],
            p["subject_type"],
            p["object_type"],
            p["subject_id"],
            p["object_id"],
        )
    )

    return selected


print("Experiment B geometry pair record helpers loaded.")

Experiment B geometry pair record helpers loaded.


In [12]:
def select_text_candidate_relations(
    relations: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Mark a subset as preferred_for_paragraph.
    Step 3 can use this as default paragraph relation pool.
    """
    for rel in relations:
        rel["preferred_for_paragraph"] = False

    candidates = [
        r for r in relations
        if r.get("candidate_for_text", True) and r.get("verbalizable", True)
    ]

    candidates.sort(
        key=lambda r: (
            -relation_priority(r),
            r["relation"],
            r["subject_type"],
            r["object_type"],
            r["subject_id"],
            r["object_id"],
        )
    )

    if MAX_TEXT_CANDIDATE_RELATIONS_PER_CLUSTER is None:
        selected = candidates
    else:
        selected = candidates[:MAX_TEXT_CANDIDATE_RELATIONS_PER_CLUSTER]

    selected_keys = {
        (
            r["subject_id"],
            r["relation"],
            r["object_id"],
        )
        for r in selected
    }

    for rel in relations:
        key = (
            rel["subject_id"],
            rel["relation"],
            rel["object_id"],
        )
        if key in selected_keys:
            rel["preferred_for_paragraph"] = True

    return relations


def build_cluster_relations(
    scene: str,
    chunk: Dict[str, Any],
    cluster: Dict[str, Any],
) -> Dict[str, Any]:
    objects = cluster.get("objects", []) or []

    topo_relations = build_topological_relations(
        scene=scene,
        chunk=chunk,
        cluster=cluster,
        objects=objects,
    )

    geo_relations = build_geometric_relations(
        scene=scene,
        chunk=chunk,
        cluster=cluster,
        objects=objects,
        topological_relations=topo_relations,
    )

    relations = topo_relations + geo_relations

    relations = deduplicate_relations(relations)
    relations = expand_inverse_relations(relations)
    relations = deduplicate_relations(relations)
    relations = compress_inverse_relations_randomly(relations)
    relations = deduplicate_relations(relations)

    relations = select_text_candidate_relations(relations)

    geometry_pairs = build_geometry_pair_records(
        scene=scene,
        chunk=chunk,
        cluster=cluster,
        objects=objects,
    )

    relation_counts = dict(Counter(r["relation"] for r in relations))
    preferred_counts = dict(
        Counter(
            r["relation"]
            for r in relations
            if r.get("preferred_for_paragraph", False)
        )
    )

    return {
        "cluster_id": cluster["cluster_id"],
        "source_chunk_id": cluster.get("source_chunk_id", chunk["chunk_id"]),
        "chunk_id": chunk["chunk_id"],
        "grid_i": chunk.get("grid_i"),
        "grid_j": chunk.get("grid_j"),
        "bounds": chunk.get("bounds"),

        "anchor_object_ids": cluster.get("anchor_object_ids", []),
        "num_objects": len(objects),
        "object_ids": cluster.get("object_ids", []),

        "num_relations": len(relations),
        "relation_counts": relation_counts,

        "num_preferred_for_paragraph": sum(
            1 for r in relations if r.get("preferred_for_paragraph", False)
        ),
        "preferred_relation_counts": preferred_counts,

        "relations": relations,

        "num_geometry_pairs": len(geometry_pairs),
        "geometry_pairs": geometry_pairs,

        "diagnostics": {
            "num_topological_relations_before_inverse": len(topo_relations),
            "num_geometric_relations_before_inverse": len(geo_relations),
            "num_objects": len(objects),
            "cluster_diagnostics_from_step1": cluster.get("diagnostics", {}),
        },
    }


print("Cluster-level processing loaded.")

Cluster-level processing loaded.


In [13]:
def build_scene_relations(scene_data: Dict[str, Any]) -> Dict[str, Any]:
    scene = scene_data["scene"]

    output_chunks = []

    all_cluster_relations = []
    all_geometry_pairs = []

    for chunk in scene_data.get("chunks", []):
        output_clusters = []

        for cluster in chunk.get("clusters", []):
            cluster_out = build_cluster_relations(
                scene=scene,
                chunk=chunk,
                cluster=cluster,
            )

            output_clusters.append(cluster_out)
            all_cluster_relations.extend(cluster_out["relations"])
            all_geometry_pairs.extend(cluster_out["geometry_pairs"])

        output_chunks.append({
            "chunk_id": chunk["chunk_id"],
            "grid_i": chunk.get("grid_i"),
            "grid_j": chunk.get("grid_j"),
            "bounds": chunk.get("bounds"),
            "num_raw_objects": chunk.get("num_raw_objects"),
            "num_clusters": len(output_clusters),
            "clusters": output_clusters,
            "diagnostics_from_step1": chunk.get("diagnostics", {}),
        })

    # Global flat relation list with cross-cluster deduplication.
    deduped_relations = global_deduplicate_relations(all_cluster_relations)
    deduped_relations = compress_inverse_relations_randomly(deduped_relations)
    deduped_relations = global_deduplicate_relations(deduped_relations)

    deduped_geometry_pairs = global_deduplicate_geometry_pairs(all_geometry_pairs)

    scene_relation_counts = dict(Counter(r["relation"] for r in deduped_relations))
    scene_preferred_counts = dict(
        Counter(
            r["relation"]
            for r in deduped_relations
            if r.get("preferred_for_paragraph", False)
        )
    )

    return {
        "scene": scene,

        "source_step1_summary": {
            "num_all_objects": scene_data.get("num_all_objects"),
            "num_valid_objects": scene_data.get("num_valid_objects"),
            "num_chunks": scene_data.get("num_chunks"),
            "num_clusters": scene_data.get("num_clusters"),
            "scene_bounds": scene_data.get("scene_bounds"),
            "chunk_config": scene_data.get("chunk_config"),
        },

        "step2_config": {
            "surface_types": sorted(SURFACE_TYPES),
            "structural_types": sorted(STRUCTURAL_TYPES),
            "inverse_relation_map": INVERSE_RELATION_MAP,
            "left_margin": LEFT_MARGIN,
            "above_y_margin": ABOVE_Y_MARGIN,
            "near_threshold": NEAR_THRESHOLD,
            "left_require_z_overlap_or_near": LEFT_REQUIRE_Z_OVERLAP_OR_NEAR,
            "left_max_z_center_distance": LEFT_MAX_Z_CENTER_DISTANCE,
            "max_near_relations_per_cluster": MAX_NEAR_RELATIONS_PER_CLUSTER,
            "max_text_candidate_relations_per_cluster": MAX_TEXT_CANDIDATE_RELATIONS_PER_CLUSTER,
            "build_geometry_pair_records": BUILD_GEOMETRY_PAIR_RECORDS,
            "max_geometry_pairs_per_cluster": MAX_GEOMETRY_PAIRS_PER_CLUSTER,
            "random_seed": RANDOM_SEED,
        },

        "num_chunks": len(output_chunks),
        "chunks": output_chunks,

        # Flat relation list for Experiment A.
        "num_cluster_relation_mentions": len(all_cluster_relations),
        "num_relations": len(deduped_relations),
        "relation_counts": scene_relation_counts,
        "preferred_relation_counts": scene_preferred_counts,
        "relations": deduped_relations,

        # Flat geometry pair list for Experiment B.
        "num_cluster_geometry_pair_mentions": len(all_geometry_pairs),
        "num_geometry_pairs": len(deduped_geometry_pairs),
        "geometry_pairs": deduped_geometry_pairs,
    }


print("Scene-level processing loaded.")

Scene-level processing loaded.


In [16]:
# ============================================================
# Run Step 2 for all available Step 1 json files
# with natural numeric sorting
# ============================================================

import re

def natural_sort_key(filename):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", filename)
    ]

expected_input_files = sorted(
    [
        filename
        for filename in os.listdir(STEP1_DIR)
        if filename.endswith(".json")
    ],
    key=natural_sort_key
)

if not expected_input_files:
    raise FileNotFoundError(
        f"No Step 1 json files found in directory:\n{STEP1_DIR}"
    )

SCENES = [
    filename.replace(".json", "")
    for filename in expected_input_files
]

print("Detected scenes:", SCENES)
print("Input files:", expected_input_files)
print("Number of input files:", len(expected_input_files))

for filename in expected_input_files:
    in_path = os.path.join(STEP1_DIR, filename)
    scene_data = load_json(in_path)

    out_data = build_scene_relations(scene_data)

    out_filename = filename.replace(".json", "_chunk_relations.json")
    out_path = os.path.join(STEP2_DIR, out_filename)

    save_json(out_path, out_data)

    print(
        f"[Saved] {out_filename} | "
        f"scene={out_data['scene']} | "
        f"relations={out_data['num_relations']} | "
        f"cluster_relation_mentions={out_data['num_cluster_relation_mentions']} | "
        f"geometry_pairs={out_data['num_geometry_pairs']} | "
        f"relation_counts={out_data['relation_counts']}"
    )

Detected scenes: ['FloorPlan1', 'FloorPlan2', 'FloorPlan3', 'FloorPlan4', 'FloorPlan5', 'FloorPlan6']
Input files: ['FloorPlan1.json', 'FloorPlan2.json', 'FloorPlan3.json', 'FloorPlan4.json', 'FloorPlan5.json', 'FloorPlan6.json']
Number of input files: 6
[Saved] FloorPlan1_chunk_relations.json | scene=FloorPlan1 | relations=308 | cluster_relation_mentions=340 | geometry_pairs=356 | relation_counts={'contains': 5, 'near': 74, 'right_of': 71, 'on': 18, 'above': 19, 'below': 22, 'supports': 17, 'left_of': 79, 'in': 3}
[Saved] FloorPlan2_chunk_relations.json | scene=FloorPlan2 | relations=221 | cluster_relation_mentions=248 | geometry_pairs=298 | relation_counts={'above': 15, 'below': 16, 'left_of': 50, 'near': 62, 'right_of': 54, 'supports': 10, 'on': 11, 'in': 3}
[Saved] FloorPlan3_chunk_relations.json | scene=FloorPlan3 | relations=143 | cluster_relation_mentions=149 | geometry_pairs=296 | relation_counts={'below': 7, 'in': 3, 'near': 46, 'above': 8, 'left_of': 36, 'on': 10, 'right_of':

In [17]:
print("STEP2_DIR:", STEP2_DIR)

print("\nOutput files:")
for name in sorted(os.listdir(STEP2_DIR)):
    print("-", name)

STEP2_DIR: /content/pilot_3/data/step2_chunk_relations

Output files:
- FloorPlan1_chunk_relations.json
- FloorPlan2_chunk_relations.json
- FloorPlan3_chunk_relations.json
- FloorPlan4_chunk_relations.json
- FloorPlan5_chunk_relations.json
- FloorPlan6_chunk_relations.json


In [18]:
from pprint import pprint

example_scene = SCENES[0]
example_filename = f"{example_scene}_chunk_relations.json"
path = os.path.join(STEP2_DIR, example_filename)

if not os.path.exists(path):
    raise FileNotFoundError(f"Missing Step 2 output file: {path}")

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=== Scene summary ===")
print("file:", example_filename)
print("scene:", data["scene"])
print("num_chunks:", data["num_chunks"])
print("num_cluster_relation_mentions:", data["num_cluster_relation_mentions"])
print("num_relations after global dedup:", data["num_relations"])
print("num_geometry_pairs:", data["num_geometry_pairs"])

print("\n=== Step 2 config saved in output ===")
pprint(data["step2_config"])

print("\n=== Relation counts ===")
pprint(data["relation_counts"])

print("\n=== Preferred relation counts ===")
pprint(data["preferred_relation_counts"])

=== Scene summary ===
file: FloorPlan1_chunk_relations.json
scene: FloorPlan1
num_chunks: 9
num_cluster_relation_mentions: 340
num_relations after global dedup: 308
num_geometry_pairs: 356

=== Step 2 config saved in output ===
{'above_y_margin': 0.03,
 'build_geometry_pair_records': True,
 'inverse_relation_map': {'above': 'below',
                          'below': 'above',
                          'contains': 'in',
                          'in': 'contains',
                          'left_of': 'right_of',
                          'near': 'near',
                          'none': 'none',
                          'on': 'supports',
                          'right_of': 'left_of',
                          'supports': 'on'},
 'left_margin': 0.05,
 'left_max_z_center_distance': 0.8,
 'left_require_z_overlap_or_near': True,
 'max_geometry_pairs_per_cluster': 80,
 'max_near_relations_per_cluster': 12,
 'max_text_candidate_relations_per_cluster': 12,
 'near_threshold': 0.5,
 'random_see

In [19]:
print("=== Cluster relation summary ===")

for chunk in data["chunks"]:
    for cluster in chunk["clusters"]:
        print(
            cluster["cluster_id"],
            "| objects =", cluster["num_objects"],
            "| relations =", cluster["num_relations"],
            "| preferred =", cluster["num_preferred_for_paragraph"],
            "| geometry_pairs =", cluster["num_geometry_pairs"],
            "| counts =", cluster["relation_counts"],
            "| preferred_counts =", cluster["preferred_relation_counts"],
        )

=== Cluster relation summary ===
FloorPlan1_chunk_0_0_cluster_0 | objects = 15 | relations = 48 | preferred = 12 | geometry_pairs = 80 | counts = {'near': 18, 'contains': 2, 'supports': 6, 'on': 3, 'above': 2, 'below': 2, 'right_of': 7, 'left_of': 8} | preferred_counts = {'contains': 2, 'supports': 6, 'on': 3, 'above': 1}
FloorPlan1_chunk_0_1_cluster_0 | objects = 15 | relations = 45 | preferred = 12 | geometry_pairs = 80 | counts = {'near': 22, 'supports': 4, 'contains': 1, 'on': 5, 'below': 2, 'left_of': 3, 'above': 1, 'right_of': 7} | preferred_counts = {'supports': 4, 'contains': 1, 'on': 5, 'above': 1, 'below': 1}
FloorPlan1_chunk_1_0_cluster_0 | objects = 15 | relations = 101 | preferred = 12 | geometry_pairs = 80 | counts = {'near': 22, 'in': 2, 'supports': 3, 'on': 2, 'below': 9, 'right_of': 29, 'left_of': 29, 'above': 5} | preferred_counts = {'in': 2, 'supports': 3, 'on': 2, 'above': 5}
FloorPlan1_chunk_1_1_cluster_0 | objects = 9 | relations = 25 | preferred = 12 | geometry_p

In [20]:
relations = data["relations"]

print("num_relations:", len(relations))

if not relations:
    print("No relations found.")
else:
    print("=== First relation ===")
    pprint(relations[0])

    print("\n=== First 10 relation labels ===")
    for rel in relations[:10]:
        d = rel["geometry_label"]["center_delta"]
        print(
            rel["cluster_id"],
            "|",
            rel["subject_type"],
            rel["relation"],
            rel["object_type"],
            "| preferred =", rel.get("preferred_for_paragraph"),
            "| dx =", round(d["dx"], 3),
            "| dy =", round(d["dy"], 3),
            "| dz =", round(d["dz"], 3),
            "| hd =", round(d["horizontal_distance"], 3),
        )

num_relations: 308
=== First relation ===
{'candidate_for_text': True,
 'chunk_id': 'FloorPlan1_chunk_0_0',
 'cluster_anchor_object_ids': ['CounterTop|-01.87|+00.95|-01.21',
                               'Cabinet|-01.55|+00.50|-01.97'],
 'cluster_id': 'FloorPlan1_chunk_0_0_cluster_0',
 'cluster_num_objects': 15,
 'derived_from_relation': 'in',
 'evidence': {'forward_evidence': {'parent_id': 'Cabinet|-01.55|+00.50|-01.97',
                                   'parent_type': 'Cabinet',
                                   'rule': 'parentReceptacles'},
              'forward_relation': 'in',
              'source': 'inverse_relation'},
 'geometry_label': {'bbox_geometry': {'has_x_overlap': True,
                                      'has_y_overlap': True,
                                      'has_z_overlap': True,
                                      'subject_bottom_minus_object_top': -0.7490800023078918,
                                      'subject_top_minus_object_bottom': 0.0270261168

In [21]:
pairs = data["geometry_pairs"]

print("num_geometry_pairs:", len(pairs))

if not pairs:
    print("No geometry pairs found.")
else:
    print("=== First geometry pair ===")
    pprint(pairs[0])

    print("\n=== First 10 geometry pairs ===")
    for pair in pairs[:10]:
        d = pair["geometry_label"]["center_delta"]
        print(
            pair["cluster_id"],
            "|",
            pair["subject_type"], "->", pair["object_type"],
            "| dx =", round(d["dx"], 3),
            "| dy =", round(d["dy"], 3),
            "| dz =", round(d["dz"], 3),
            "| hd =", round(d["horizontal_distance"], 3),
        )

num_geometry_pairs: 356
=== First geometry pair ===
{'candidate_for_geometry_regression': True,
 'chunk_id': 'FloorPlan1_chunk_0_0',
 'cluster_anchor_object_ids': ['CounterTop|-01.87|+00.95|-01.21',
                               'Cabinet|-01.55|+00.50|-01.97'],
 'cluster_id': 'FloorPlan1_chunk_0_0_cluster_0',
 'cluster_num_objects': 15,
 'geometry_label': {'bbox_geometry': {'has_x_overlap': True,
                                      'has_y_overlap': True,
                                      'has_z_overlap': True,
                                      'subject_bottom_minus_object_top': -0.027026116847991943,
                                      'subject_top_minus_object_bottom': 0.7490800023078918,
                                      'x_overlap': 0.1359119415283203,
                                      'y_overlap': 0.026105523109436035,
                                      'z_overlap': 0.15683388710021973},
                    'center_delta': {'dx': 0.30676162242889404,
       

In [22]:
for scene in SCENES:
    filename = f"{scene}_chunk_relations.json"
    path = os.path.join(STEP2_DIR, filename)

    if not os.path.exists(path):
        print("\nMissing:", filename)
        continue

    with open(path, "r", encoding="utf-8") as f:
        scene_out = json.load(f)

    labels = [r["relation"] for r in scene_out["relations"]]
    preferred_labels = [
        r["relation"] for r in scene_out["relations"]
        if r.get("preferred_for_paragraph", False)
    ]

    print("\nFile:", filename)
    print("scene:", scene_out["scene"])
    print("num_relations:", len(labels))
    print("label counts:", dict(Counter(labels)))
    print("preferred label counts:", dict(Counter(preferred_labels)))
    print("num_geometry_pairs:", scene_out["num_geometry_pairs"])


File: FloorPlan1_chunk_relations.json
scene: FloorPlan1
num_relations: 308
label counts: {'contains': 5, 'near': 74, 'right_of': 71, 'on': 18, 'above': 19, 'below': 22, 'supports': 17, 'left_of': 79, 'in': 3}
preferred label counts: {'contains': 5, 'on': 18, 'above': 10, 'supports': 17, 'below': 2, 'in': 3, 'left_of': 7}
num_geometry_pairs: 356

File: FloorPlan2_chunk_relations.json
scene: FloorPlan2
num_relations: 221
label counts: {'above': 15, 'below': 16, 'left_of': 50, 'near': 62, 'right_of': 54, 'supports': 10, 'on': 11, 'in': 3}
preferred label counts: {'above': 11, 'below': 8, 'supports': 10, 'on': 11, 'left_of': 4, 'right_of': 1, 'in': 3}
num_geometry_pairs: 298

File: FloorPlan3_chunk_relations.json
scene: FloorPlan3
num_relations: 143
label counts: {'below': 7, 'in': 3, 'near': 46, 'above': 8, 'left_of': 36, 'on': 10, 'right_of': 27, 'supports': 6}
preferred label counts: {'below': 5, 'in': 3, 'near': 4, 'above': 8, 'left_of': 25, 'on': 10, 'supports': 6, 'right_of': 5}
num

In [23]:
import shutil

zip_base = "/content/pilot3_step2_chunk_relations"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    STEP2_DIR,
)

print("Created:", zip_path)

Created: /content/pilot3_step2_chunk_relations.zip


In [24]:
from google.colab import files

files.download("/content/pilot3_step2_chunk_relations.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>